In [26]:
import polars as pl
import datetime
from polars import DataFrame

In [27]:
dataset = '../data/HI-Medium_Trans.csv'

trans = (
    pl.read_csv(dataset)
    .with_columns(
        pl.col("Timestamp").str.strptime(pl.Datetime, '%Y/%m/%d %H:%M', strict=True)
    )
    .sort('Timestamp')
)

In [28]:
trans.head()

Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-09-01 00:00:00,1046,"""800A37D90""",274159,"""820C04F20""",26.42,"""US Dollar""",26.42,"""US Dollar""","""Credit Card""",0
2022-09-01 00:00:00,21418,"""800AB4DE0""",21418,"""800AB4DE0""",23.61,"""US Dollar""",23.61,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,32248,"""80056BBD0""",11,"""800BAFE20""",12373.74,"""US Dollar""",12373.74,"""US Dollar""","""ACH""",0
2022-09-01 00:00:00,11798,"""80145BAF0""",11798,"""80145BAF0""",4981.27,"""US Dollar""",4981.27,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,1924,"""800CCE420""",1924,"""800CCE420""",22.94,"""US Dollar""",22.94,"""US Dollar""","""Reinvestment""",0


In [29]:
zero = trans['Timestamp'].min()
hundred = trans['Timestamp'].max()

In [30]:
diff = hundred - zero
days = diff.days
sixty = zero + datetime.timedelta(days=days * 0.6)
eighty = zero + datetime.timedelta(days=days * 0.8)
hundred = zero + datetime.timedelta(days=days)

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-17 04:48:00 2022-09-22 14:24:00 2022-09-28 00:00:00


# Format data to remove strings

In [31]:
nodes = (
    trans
    .select(pl.col('From').alias('Acc'))
    .vstack(trans.select(pl.col('To').alias('Acc')))
    .unique()
    .with_row_index('Node ID')
)

payment_formats = (
    trans
    .select(pl.col('Payment Format'))
    .unique()
    .with_row_index('format_id')
)

currencies = (
    trans
    .select(pl.col('Receiving Currency').alias('Currency'))
    .vstack(trans.select(pl.col('Payment Currency').alias('Currency')))
    .unique()
    .with_row_index('currency_id')
)

ssl = (
    trans.join(nodes, left_on='From', right_on='Acc')
    .join(nodes, left_on='To', right_on='Acc', suffix='_to')
    .join(payment_formats, on='Payment Format')
    .join(currencies, left_on='Payment Currency', right_on='Currency')
    .join(currencies, left_on='Receiving Currency', right_on='Currency', suffix='_to')
    .drop('From', 'To', 'Payment Format', 'Payment Currency', 'Receiving Currency')
    .rename({
        'Node ID': 'From',
        'Node ID_to': 'To',
        'format_id': 'Payment Format',
        'currency_id': 'Payment Currency',
        'currency_id_to': 'Receiving Currency',
    })
)

In [32]:
ssl.head()

Timestamp,From Bank,To Bank,Amount Received,Amount Paid,Is Laundering,From,To,Payment Format,Payment Currency,Receiving Currency
datetime[μs],i64,i64,f64,f64,i64,u32,u32,u32,u32,u32
2022-09-01 00:00:00,1046,274159,26.42,26.42,0,1635842,1600487,1,9,9
2022-09-01 00:00:00,21418,21418,23.61,23.61,0,798353,798353,2,9,9
2022-09-01 00:00:00,32248,11,12373.74,12373.74,0,1748257,799071,6,9,9
2022-09-01 00:00:00,11798,11798,4981.27,4981.27,0,1630856,1630856,2,9,9
2022-09-01 00:00:00,1924,1924,22.94,22.94,0,499983,499983,2,9,9


# Global vars

In [33]:
every = '2d'

In [34]:
def _prep(df: pl.DataFrame) -> pl.LazyFrame:
    return (
        df.lazy()
        .with_columns(
            pl.col('Timestamp').dt.truncate(every).alias("window_start")
        )
        .filter(
             pl.col('Timestamp').is_not_null()
             & pl.col('From').is_not_null()
             & pl.col('To').is_not_null()
        )
    )

In [35]:
_prep(ssl).sort(pl.col('window_start')).collect()

Timestamp,From Bank,To Bank,Amount Received,Amount Paid,Is Laundering,From,To,Payment Format,Payment Currency,Receiving Currency,window_start
datetime[μs],i64,i64,f64,f64,i64,u32,u32,u32,u32,u32,datetime[μs]
2022-09-01 00:00:00,1046,274159,26.42,26.42,0,1635842,1600487,1,9,9,2022-09-01 00:00:00
2022-09-01 00:00:00,21418,21418,23.61,23.61,0,798353,798353,2,9,9,2022-09-01 00:00:00
2022-09-01 00:00:00,32248,11,12373.74,12373.74,0,1748257,799071,6,9,9,2022-09-01 00:00:00
2022-09-01 00:00:00,11798,11798,4981.27,4981.27,0,1630856,1630856,2,9,9,2022-09-01 00:00:00
2022-09-01 00:00:00,1924,1924,22.94,22.94,0,499983,499983,2,9,9,2022-09-01 00:00:00
…,…,…,…,…,…,…,…,…,…,…,…
2022-09-28 11:59:00,211767,219853,7339.28,7339.28,1,954955,1051471,6,6,6,2022-09-27 00:00:00
2022-09-28 12:14:00,211767,211767,5869.16,6877.38,0,954955,954955,6,9,2,2022-09-27 00:00:00
2022-09-28 12:14:00,211767,130596,5869.16,5869.16,1,954955,482179,6,2,2,2022-09-27 00:00:00


# Temporal stats

In [36]:
def add_temporal_stats(df: pl.DataFrame) -> pl.DataFrame:
    df = _prep(df).select(
        pl.col('window_start'),
        pl.col('Timestamp'),
        pl.col('From'),
        pl.col('To'),
    )

    in_gaps = (
        df.sort(["window_start", "To", "Timestamp"])
        .group_by(["window_start", "To"])
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_in_sec"),
            pl.col("gap").list.var().alias("var_gap_in_sec"),
        ])
        .select(["window_start", 'To', "mu_gap_in_sec", "var_gap_in_sec"])
        .rename({'To': 'Node'})
    )

    out_gaps = (
        df.sort(["window_start", "From", "Timestamp"])
        .group_by(["window_start", "From"])
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_out_sec"),
            pl.col("gap").list.var().alias("var_gap_out_sec"),
        ])
        .select(["window_start", 'From', "mu_gap_out_sec", "var_gap_out_sec"])
        .rename({'From': 'Node'})
    )

    return (
        in_gaps.join(out_gaps, on=["Node", 'window_start'], how="full", coalesce=True)
        .collect()
    )

In [37]:
temporal_features = add_temporal_stats(ssl)

In [38]:
temporal_features

window_start,Node,mu_gap_in_sec,var_gap_in_sec,mu_gap_out_sec,var_gap_out_sec
datetime[μs],u32,f64,f64,f64,f64
2022-09-11 00:00:00,19246,300.0,null,null,null
2022-09-11 00:00:00,1732155,420.0,null,null,null
2022-09-01 00:00:00,166312,null,null,null,null
2022-09-09 00:00:00,1197691,3440.0,2.91252e7,37425.0,2.5486e9
2022-09-13 00:00:00,169900,13872.0,8.8880832e8,null,null
…,…,…,…,…,…
2022-09-15 00:00:00,1302739,null,null,null,null
2022-09-03 00:00:00,1557619,null,null,40060.0,4.6571e9
2022-09-07 00:00:00,1112333,null,null,8305.263158,1.1227e8


# Fraudulent patterns

## 2-cycles

In [39]:
def count_reciprocal_neighbours(df: DataFrame) -> DataFrame:
    lf = _prep(df)

    edges = lf.select(["window_start", "From", "To"]).unique().filter(pl.col("From") != pl.col("To"))

    recip = (
        edges.join(
            edges,
            how="inner",
            left_on=["window_start", "From", "To"],
            right_on=["window_start", "To", "From"],
        )
        .select(["window_start", "From", "To"])
        .unique()
        .group_by(["window_start", "From"])
        .agg(
            pl.col("To").n_unique().alias("r_2cycle")
        )
        .rename({"From": "Node"})
    )

    nodes = (
        pl.concat([
            lf.select(["window_start", pl.col("From").alias("Node")]),
            lf.select(["window_start", pl.col("To").alias("Node")]),
        ])
        .unique()
    )

    return (
        nodes.join(recip, on=["window_start", "Node"], how="left")
        .with_columns(
            pl.col("r_2cycle").fill_null(0).cast(pl.Int64)
        )
        .collect()
    )


In [40]:
two_cycles = count_reciprocal_neighbours(ssl)

In [41]:
two_cycles

window_start,Node,r_2cycle
datetime[μs],u32,i64
2022-09-01 00:00:00,791995,0
2022-09-03 00:00:00,626812,0
2022-09-01 00:00:00,1159975,0
2022-09-15 00:00:00,1977113,0
2022-09-15 00:00:00,1197935,0
…,…,…
2022-09-15 00:00:00,1201715,0
2022-09-15 00:00:00,694014,0
2022-09-09 00:00:00,513563,0


In [42]:
two_cycles.describe()

statistic,window_start,Node,r_2cycle
str,str,f64,f64
"""count""","""9337964""",9.337964e6,9.337964e6
"""null_count""","""0""",0.0,0.0
"""mean""","""2022-09-07 18:20:54.673845""",1.0384e6,0.001221
"""std""",null,599830.206456,0.035827
"""min""","""2022-09-01 00:00:00""",0.0,0.0
"""25%""","""2022-09-03 00:00:00""",518769.0,0.0
"""50%""","""2022-09-09 00:00:00""",1.038385e6,0.0
"""75%""","""2022-09-13 00:00:00""",1.557822e6,0.0
"""max""","""2022-09-27 00:00:00""",2.076998e6,3.0


## Ego profile

In [43]:
EPS = 1e-12

def compute_ego_profiles(trans: DataFrame) -> DataFrame:
    lf = _prep(trans)

    out_stats = (
        lf.group_by(["window_start", "From"])
        .agg(
            pl.len().alias("deg_out"),
            pl.col("To").n_unique().alias("fan_out"),
            pl.col("Amount Paid").sum().alias("vol_out"),
        )
        .rename({"From": "Node"})
    )

    in_stats = (
        lf.group_by(["window_start", "To"])
        .agg(
            pl.len().alias("deg_in"),
            pl.col("From").n_unique().alias("fan_in"),
            pl.col("Amount Received").sum().alias("vol_in"),
        )
        .rename({"To": "Node"})
    )

    ego = (
        out_stats.join(in_stats, on=["window_start", "Node"], how="full")
        .with_columns(
            pl.col("deg_out").fill_null(0).cast(pl.Int64),
            pl.col("fan_out").fill_null(0).cast(pl.Int64),
            pl.col("vol_out").fill_null(0.0),
            pl.col("deg_in").fill_null(0).cast(pl.Int64),
            pl.col("fan_in").fill_null(0).cast(pl.Int64),
            pl.col("vol_in").fill_null(0.0),
        )
        .with_columns(
            ((pl.col("vol_in") - pl.col("vol_out"))/ (pl.col("vol_in") + pl.col("vol_out") + pl.lit(EPS))).alias("flow_imbalance")
        )
        .with_columns(
            pl.when(pl.col('window_start').is_null()).then(pl.col('window_start_right')).otherwise(pl.col('window_start')).alias('window_start'),
            pl.when(pl.col('Node').is_null()).then(pl.col('Node_right')).otherwise(pl.col('Node')).alias('Node'),
        )
        .drop('Node_right', 'window_start_right')
    )

    per_cur_in = (
        lf.group_by(["window_start", "To", "Receiving Currency"])
        .agg(pl.col("Amount Received").sum().alias("vol_in_cur"))
        .rename({"To": "Node", "Receiving Currency": "currency"})
    )

    mix_in = (
        per_cur_in.join(
            per_cur_in.group_by(["window_start", "Node"])
            .agg(pl.col("vol_in_cur").sum().alias("vol_in_sum")),
            on=["window_start", "Node"],
        )
        .with_columns((pl.col("vol_in_cur") / (pl.col("vol_in_sum") + pl.lit(EPS))).alias("p"))
        .group_by(["window_start", "Node"])
        .agg([
            pl.len().alias("n_currencies_in"),
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("currency_entropy_in"),
            pl.col("p").max().alias("top_currency_share_in"),
        ])
    )

    per_cur_out = (
        lf.group_by(["window_start", "From", "Payment Currency"])
        .agg(pl.col("Amount Paid").sum().alias("vol_out_cur"))
        .rename({"From": "Node", "Payment Currency": "currency"})
    )

    mix_out = (
        per_cur_out.join(
            per_cur_out.group_by(["window_start", "Node"])
            .agg(pl.col("vol_out_cur").sum().alias("vol_out_sum")),
            on=["window_start", "Node"],
        )
        .with_columns((pl.col("vol_out_cur") / (pl.col("vol_out_sum") + pl.lit(EPS))).alias("p"))
        .group_by(["window_start", "Node"])
        .agg([
            pl.len().alias("n_currencies_out"),
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("currency_entropy_out"),
            pl.col("p").max().alias("top_currency_share_out"),
        ])
    )

    full = (
        ego.join(mix_in, on=["window_start", "Node"], how="left")
        .join(mix_out, on=["window_start", "Node"], how="left")
        .with_columns(
            pl.col("n_currencies_in").fill_null(0).cast(pl.Int64),
            pl.col("currency_entropy_in").fill_null(0.0),
            pl.col("top_currency_share_in").fill_null(0.0),

            pl.col("n_currencies_out").fill_null(0).cast(pl.Int64),
            pl.col("currency_entropy_out").fill_null(0.0),
            pl.col("top_currency_share_out").fill_null(0.0),
        )
    )

    return full.collect()

In [44]:
ego_profile = compute_ego_profiles(ssl)

In [45]:
ego_profile

window_start,Node,deg_out,fan_out,vol_out,deg_in,fan_in,vol_in,flow_imbalance,n_currencies_in,currency_entropy_in,top_currency_share_in,n_currencies_out,currency_entropy_out,top_currency_share_out
datetime[μs],u32,i64,i64,f64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64
2022-09-11 00:00:00,1045118,2,1,1541.47,4,1,26495.66,0.890041,1,-1.0001e-12,1.0,1,-9.9942e-13,1.0
2022-09-01 00:00:00,1484569,2,1,106489.78,13,4,128806.02,0.094843,1,-1.0001e-12,1.0,1,-1.0001e-12,1.0
2022-09-03 00:00:00,527871,0,0,0.0,4,1,4297.92,1.0,1,-9.9987e-13,1.0,0,0.0,0.0
2022-09-07 00:00:00,450588,0,0,0.0,2,1,8258.02,1.0,1,-9.9987e-13,1.0,0,0.0,0.0
2022-09-11 00:00:00,76002,2,1,2.8025e6,3,1,1.4705e7,0.679841,1,-1.0001e-12,1.0,1,-1.0001e-12,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2022-09-01 00:00:00,847396,1,1,131.45,0,0,0.0,-1.0,0,0.0,0.0,1,-9.9254e-13,1.0
2022-09-09 00:00:00,1952320,1,1,1822.12,0,0,0.0,-1.0,0,0.0,0.0,1,-9.9964e-13,1.0
2022-09-15 00:00:00,41740,1,1,620.41,0,0,0.0,-1.0,0,0.0,0.0,1,-9.9831e-13,1.0


# Flow-aware positional prediction

In [46]:
EPS = 1e-12

def flow_targets_out_entropy_count(trans: DataFrame, k2: bool = True) -> DataFrame:
    lf = _prep(trans).select(["window_start", "From", "To"])

    # Edge weight = count of transactions (currency invariant)
    W = (
        lf.group_by(["window_start", "From", "To"])
        .agg(pl.len().alias("w"))
    )

    out_sum = (
        W.group_by(["window_start", "From"])
        .agg(pl.col("w").sum().alias("out_wsum"))
    )

    P = (
        W.join(out_sum, on=["window_start", "From"], how="left")
        .with_columns((pl.col("w") / (pl.col("out_wsum") + pl.lit(EPS))).alias("p"))
        .select(["window_start", "From", "To", "p"])
    )

    one = (
        P.group_by(["window_start", "From"])
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_1_cnt"),
            pl.len().alias("supp_out_1_cnt"),
            pl.col("p").max().alias("pmax_out_1_cnt"),
        )
        .rename({"From": "Node"})
    )

    if not k2:
        return one.collect()

    # 2-step distribution via mid join: sum_mid P(src->mid)*P(mid->dst)
    P1 = P.rename({"From": "src", "To": "mid", "p": "p1"})
    P2 = P.rename({"From": "mid", "To": "dst", "p": "p2"})

    two_step = (
        P1.join(P2, on=["window_start", "mid"], how="inner")
        .with_columns((pl.col("p1") * pl.col("p2")).alias("p2step"))
        .group_by(["window_start", "src", "dst"])
        .agg(pl.col("p2step").sum().alias("p"))
    )

    two = (
        two_step.group_by(["window_start", "src"])
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_2_cnt"),
            pl.len().alias("supp_out_2_cnt"),
            pl.col("p").max().alias("pmax_out_2_cnt"),
        )
        .rename({"src": "Node"})
    )

    return one.join(two, on=["window_start", "Node"], how="left").collect()


def currency_mix_out(trans: DataFrame) -> DataFrame:
    lf = _prep(trans).select(["window_start", "From", "Payment Currency", "Amount Paid"])

    per_cur = (
        lf.group_by(["window_start", "From", "Payment Currency"])
        .agg(pl.col("Amount Paid").sum().alias("vol_out_cur"))
        .rename({"From": "Node", "Payment Currency": "currency"})
    )

    mix = (
        per_cur.join(
            per_cur.group_by(["window_start", "Node"])
            .agg(pl.col("vol_out_cur").sum().alias("vol_out_sum")),
            on=["window_start", "Node"],
        )
        .with_columns((pl.col("vol_out_cur") / (pl.col("vol_out_sum") + pl.lit(EPS))).alias("p"))
        .group_by(["window_start", "Node"])
        .agg([
            pl.count().alias("n_currencies_out"),
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("currency_entropy_out"),
            pl.col("p").max().alias("top_currency_share_out"),
        ])
    )

    return mix.collect()


def flow_heads_A_B(trans: DataFrame, k2: bool = True) -> DataFrame:
    head_a = flow_targets_out_entropy_count(trans, k2=k2)   # (window_start, node, ...)
    head_b = currency_mix_out(trans)                        # (window_start, node, ...)

    # Node universe: all active nodes in the window (senders or receivers)
    lf = _prep(trans).select(["window_start", "From", "To"])
    nodes = (
        pl.concat([
            lf.select(["window_start", pl.col("From").alias("Node")]),
            lf.select(["window_start", pl.col("To").alias("Node")]),
        ])
        .unique()
        .collect()
    )

    full = (
        nodes.lazy()
        .join(head_a.lazy(), on=["window_start", "Node"], how="left")
        .join(head_b.lazy(), on=["window_start", "Node"], how="left")
        .with_columns(
            # Fill missing: nodes with no outgoing edges => no flow / no currency mix
            pl.col("H_out_1_cnt").fill_null(0.0),
            pl.col("supp_out_1_cnt").fill_null(0).cast(pl.Int64),
            pl.col("pmax_out_1_cnt").fill_null(0.0),

            pl.col("H_out_2_cnt").fill_null(0.0),
            pl.col("supp_out_2_cnt").fill_null(0).cast(pl.Int64),
            pl.col("pmax_out_2_cnt").fill_null(0.0),

            pl.col("n_currencies_out").fill_null(0).cast(pl.Int64),
            pl.col("currency_entropy_out").fill_null(0.0),
            pl.col("top_currency_share_out").fill_null(0.0),
        )
        .collect()
    )

    return full


In [47]:
# train_pos_pred = flow_heads_A_B(ssl)

# Join

In [48]:
node_features = (
    temporal_features
    # .join(train_pos_pred, on=["window_start", "Node"], how="full", coalesce=True)
    .join(ego_profile, on=["window_start", "Node"], how="full", coalesce=True)
    .join(two_cycles, on=["window_start", "Node"], how="full", coalesce=True)
    .with_columns(
        (pl.col("window_start") - pl.col("window_start").min())
        .dt.total_seconds()
        .cast(pl.Int64)
        .add(10)
        .alias("window_start"),

        pl.lit(1).alias("Feature"),
    )
    .sort("window_start")
)
node_features

window_start,Node,mu_gap_in_sec,var_gap_in_sec,mu_gap_out_sec,var_gap_out_sec,deg_out,fan_out,vol_out,deg_in,fan_in,vol_in,flow_imbalance,n_currencies_in,currency_entropy_in,top_currency_share_in,n_currencies_out,currency_entropy_out,top_currency_share_out,r_2cycle,Feature
i64,u32,f64,f64,f64,f64,i64,i64,f64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,i32
10,791995,20180.0,9.3695568e8,null,null,1,1,8.86,7,3,7104.64,0.997509,1,-9.9987e-13,1.0,1,-8.8707e-13,1.0,0,1
10,1159975,15630.0,3.418692e8,null,null,1,1,90.19,5,2,576.65,0.7295,1,-9.9831e-13,1.0,1,-9.8899e-13,1.0,0,1
10,482968,12997.5,5.6714805e8,57240.0,null,2,2,9397.98,9,2,56541.38,0.714951,1,-1.0001e-12,1.0,1,-9.9987e-13,1.0,0,1
10,297314,null,null,null,null,1,1,34867.68,1,1,34867.68,0.0,1,-1.0001e-12,1.0,1,-1.0001e-12,1.0,0,1
10,1132308,null,null,29010.0,1.6762e9,3,3,1.0883e6,1,1,623609.47,-0.271447,1,-1.0001e-12,1.0,1,-1.0001e-12,1.0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2246410,697895,null,null,0.0,null,2,2,104222.03,1,1,13539.55,-0.770051,1,-9.9987e-13,1.0,2,0.386217,0.870089,0,1
2246410,871229,13560.0,null,4520.0,6.12912e7,4,3,78485.52,2,1,67563.07,-0.074786,2,0.190095,0.95283,3,0.567086,0.82023,0,1
2246410,954955,32700.0,1.9947e9,12262.5,8.5632e8,9,5,166084.42,4,1,118652.97,-0.16658,2,0.556704,0.755063,3,0.995869,0.539427,0,1


In [49]:
trans = (
    ssl.with_columns(
        (pl.col("Timestamp") - pl.col("Timestamp").min())
        .dt.total_seconds()
        .cast(pl.Int64)
        .add(10)
        .alias("Timestamp"),
    )
    .sort("Timestamp")
    .with_row_index("Edge IDq")
)
trans

Edge IDq,Timestamp,From Bank,To Bank,Amount Received,Amount Paid,Is Laundering,From,To,Payment Format,Payment Currency,Receiving Currency
u32,i64,i64,i64,f64,f64,i64,u32,u32,u32,u32,u32
0,10,1046,274159,26.42,26.42,0,1635842,1600487,1,9,9
1,10,21418,21418,23.61,23.61,0,798353,798353,2,9,9
2,10,32248,11,12373.74,12373.74,0,1748257,799071,6,9,9
3,10,11798,11798,4981.27,4981.27,0,1630856,1630856,2,9,9
4,10,1924,1924,22.94,22.94,0,499983,499983,2,9,9
…,…,…,…,…,…,…,…,…,…,…,…
31898233,2375950,211767,219853,7339.28,7339.28,1,954955,1051471,6,6,6
31898234,2376850,211767,211767,5869.16,6877.38,0,954955,954955,6,9,2
31898235,2376850,211767,130596,5869.16,5869.16,1,954955,482179,6,2,2


# Write

In [50]:
node_features.write_csv('../data/HI-Medium_SSL_Nodes.csv')
trans.write_csv('../data/HI-Medium_SSL_Trans.csv')